# Website A/B Testing - Lab

## Introduction

In this lab, you'll get another chance to practice your skills at conducting a full A/B test analysis. It will also be a chance to practice your data exploration and processing skills! The scenario you'll be investigating is data collected from the homepage of a music app page for audacity.

## Objectives

You will be able to:
* Analyze the data from a website A/B test to draw relevant conclusions
* Explore and analyze web action data

## Exploratory Analysis

Start by loading in the dataset stored in the file 'homepage_actions.csv'. Then conduct an exploratory analysis to get familiar with the data.

> Hints:
    * Start investigating the id column:
        * How many viewers also clicked?
        * Are there any anomalies with the data; did anyone click who didn't view?
        * Is there any overlap between the control and experiment groups? 
            * If so, how do you plan to account for this in your experimental design?

In [1]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chi2_contingency, norm



# Load the dataset
df = pd.read_csv(r'C:\\Users\\USER\\Documents\\Flatiron\\Phase2\\mini assignments\\dsc-website-ab-testing-lab-master\\homepage_actions.csv')


In [2]:
# Dataset overview
df.info()

# Unique values in key columns
print(df['id'].nunique())  # Check unique viewers
print(df['action'].value_counts())  # Count of each action
print(df['group'].value_counts())  # Check group distribution


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8188 entries, 0 to 8187
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   timestamp  8188 non-null   object
 1   id         8188 non-null   int64 
 2   group      8188 non-null   object
 3   action     8188 non-null   object
dtypes: int64(1), object(3)
memory usage: 256.0+ KB
6328
action
view     6328
click    1860
Name: count, dtype: int64
group
control       4264
experiment    3924
Name: count, dtype: int64


In [3]:
# Viewers who clicked
viewers_with_clicks = df[df['action'] == 'click']['id'].nunique()

# Validate if there are clicks without views
all_viewers = set(df[df['action'] == 'view']['id'])
all_clickers = set(df[df['action'] == 'click']['id'])

# Overlap between control and experiment groups
control_ids = set(df[df['group'] == 'control']['id'])
experiment_ids = set(df[df['group'] == 'experiment']['id'])

overlap_ids = control_ids & experiment_ids
print(f'Overlap IDs count: {len(overlap_ids)}')


Overlap IDs count: 0


## Conduct a Statistical Test

Conduct a statistical test to determine whether the experimental homepage was more effective than that of the control group.

In [4]:
# Prepare data for statistical testing

# Aggregate data by group and calculate conversion rates
conversion_data = df.groupby(['group', 'action']).size().unstack(fill_value=0)
conversion_data['conversion_rate'] = conversion_data['click'] / conversion_data['view']
print(conversion_data)

# Contingency table for chi-squared test
contingency_table = [
    [conversion_data.loc['control', 'click'], conversion_data.loc['control', 'view'] - conversion_data.loc['control', 'click']],
    [conversion_data.loc['experiment', 'click'], conversion_data.loc['experiment', 'view'] - conversion_data.loc['experiment', 'click']],
]

# Chi-squared test
chi2, p_chi2, _, _ = chi2_contingency(contingency_table)
print(f'Chi-squared test p-value: {p_chi2}')

# Z-test for proportions
successes = [conversion_data.loc['control', 'click'], conversion_data.loc['experiment', 'click']]
trials = [conversion_data.loc['control', 'view'], conversion_data.loc['experiment', 'view']]

z_stat, p_ztest = proportions_ztest(successes, trials, alternative='two-sided')
print(f'Z-test p-value: {p_ztest}')

# Interpret the results
alpha = 0.05
if p_ztest < alpha:
    print("Reject the null hypothesis: The experimental homepage is more effective.")
else:
    print("Fail to reject the null hypothesis: No significant difference.")


action      click  view  conversion_rate
group                                   
control       932  3332         0.279712
experiment    928  2996         0.309746
Chi-squared test p-value: 0.009571680497042271
Z-test p-value: 0.008830075576595804
Reject the null hypothesis: The experimental homepage is more effective.


## Verifying Results

One sensible formulation of the data to answer the hypothesis test above would be to create a binary variable representing each individual in the experiment and control group. This binary variable would represent whether or not that individual clicked on the homepage; 1 for they did and 0 if they did not. 

The variance for the number of successes in a sample of a binomial variable with n observations is given by:

## $n\bullet p (1-p)$

Given this, perform 3 steps to verify the results of your statistical test:
1. Calculate the expected number of clicks for the experiment group, if it had the same click-through rate as that of the control group. 
2. Calculate the number of standard deviations that the actual number of clicks was from this estimate. 
3. Finally, calculate a p-value using the normal distribution based on this z-score.

### Step 1:
Calculate the expected number of clicks for the experiment group, if it had the same click-through rate as that of the control group. 

In [5]:
# Step 1: Calculate the expected number of clicks for the experiment group
control_rate = conversion_data.loc['control', 'conversion_rate']
experiment_views = conversion_data.loc['experiment', 'view']
expected_experiment_clicks = experiment_views * control_rate
print(f"Expected clicks for experiment group: {expected_experiment_clicks}")

Expected clicks for experiment group: 838.0168067226891


### Step 2:
Calculate the number of standard deviations that the actual number of clicks was from this estimate.

In [6]:
# Step 2: Calculate the number of standard deviations from the actual number of clicks
actual_experiment_clicks = conversion_data.loc['experiment', 'click']
std_dev = (experiment_views * control_rate * (1 - control_rate)) ** 0.5
z_score = (actual_experiment_clicks - expected_experiment_clicks) / std_dev
print(f"Z-score: {z_score}")

Z-score: 3.6625360854823588


### Step 3: 
Finally, calculate a p-value using the normal distribution based on this z-score.

In [7]:
# Step 3: Calculate the p-value using the normal distribution
p_value_normal = 2 * (1 - norm.cdf(abs(z_score)))
print(f"P-value (normal distribution): {p_value_normal}")


P-value (normal distribution): 0.0002497305601389943


In [8]:
# Analysis
if p_value_normal < alpha:
    print("The result matches the statistical test: The experimental homepage is more effective.")
else:
    print("The result matches the statistical test: No significant difference between groups.")


The result matches the statistical test: The experimental homepage is more effective.


### Analysis:

Does this result roughly match that of the previous statistical test?

> Comment: **yes it does**

## Summary

In this lab, you continued to get more practice designing and conducting AB tests. This required additional work preprocessing and formulating the initial problem in a suitable manner. Additionally, you also saw how to verify results, strengthening your knowledge of binomial variables, and reviewing initial statistical concepts of the central limit theorem, standard deviation, z-scores, and their accompanying p-values.